In [ ]:
import re
from IPython.display import display

from qniverse.QniverseCircuit import *


def qasm_to_qniverse(qasm_code, qasm_version):

    errors = []

    # -----------------------
    # Validate QASM Version
    # -----------------------

    if qasm_version not in ["2.0", "3.0"]:
        errors.append(
            f"Unsupported QASM version: {qasm_version}"
        )
        return None, errors

    gate_map = {
        "h": h,
        "x": x,
        "y": y,
        "z": z,
        "cx": cx
    }

    lines = qasm_code.splitlines()

    qc = QniverseCircuit()

    # -----------------------
    # Detect unsupported dynamic constructs
    # -----------------------

    dynamic_constructs = [
        "if",
        "while",
        "for",
        "switch"
    ]

    for line_no, line in enumerate(lines, start=1):

        stripped = line.strip()

        for construct in dynamic_constructs:

            if stripped.startswith(construct):

                errors.append(
                    f"Line {line_no}: Unsupported construct '{construct}'"
                )

    # -----------------------
    # Determine qubit count
    # -----------------------

    max_qubit = -1

    for line in lines:

        matches = re.findall(
            r'q\[(\d+)\]',
            line
        )

        for m in matches:
            max_qubit = max(
                max_qubit,
                int(m)
            )

    if max_qubit == -1:

        errors.append(
            "No qubits found in source"
        )

        return None, errors

    num_qubits = max_qubit + 1

    try:

        q = QuantumRegister(num_qubits,'q')
        qc.add_register(q)

    except Exception as e:

        errors.append(
            f"Register creation failed: {e}"
        )

        return None, errors

    # -----------------------
    # Parse instructions
    # -----------------------

    for line_no, line in enumerate(lines, start=1):

        line = line.strip()

        if (
            not line
            or line.startswith("OPENQASM")
            or line.startswith("include")
            or line.startswith("qreg")
            or line.startswith("creg")
            or line.startswith("//")
        ):
            continue

        line = line.rstrip(";")

        # -------------------
        # CX Gate
        # -------------------

        if line.startswith("cx"):

            match = re.findall(
                r'q\[(\d+)\]',
                line
            )

            if len(match) != 2:

                errors.append(
                    f"Line {line_no}: Malformed CX gate"
                )

                continue

            control = int(match[0])
            target = int(match[1])

            if (
                control >= num_qubits
                or target >= num_qubits
            ):

                errors.append(
                    f"Line {line_no}: Invalid qubit index"
                )

                continue

            qc.add_gate(
                cx,
                q[control],
                q[target]
            )

            continue

        elif line.startswith("h"):

            match = re.findall(
                r'q\[(\d+)\]',
                line
            )

            if len(match) != 1:

                errors.append(
                    f"Line {line_no}: Malformed H gate"
                )

                continue

            target = int(match[0])

            if target >= num_qubits:

                errors.append(
                    f"Line {line_no}: Invalid qubit index"
                )

                continue

            qc.add_gate(
                h,
                q[target]
            )

            continue
            
        elif line.startswith("x"):

            match = re.findall(
                r'q\[(\d+)\]',
                line
            )

            if len(match) != 1:

                errors.append(
                    f"Line {line_no}: Malformed X gate"
                )

                continue

            target = int(match[0])

            if target >= num_qubits:

                errors.append(
                    f"Line {line_no}: Invalid qubit index"
                )

                continue

            qc.add_gate(
                x,
                q[target]
            )

            continue
            
        elif line.startswith("y"):

            match = re.findall(
                r'q\[(\d+)\]',
                line
            )

            if len(match) != 1:

                errors.append(
                    f"Line {line_no}: Malformed Y gate"
                )

                continue

            target = int(match[0])

            if target >= num_qubits:

                errors.append(
                    f"Line {line_no}: Invalid qubit index"
                )

                continue

            qc.add_gate(
                y,
                q[target]
            )

            continue
            
        elif line.startswith("z"):

            match = re.findall(
                r'q\[(\d+)\]',
                line
            )

            if len(match) != 1:

                errors.append(
                    f"Line {line_no}: Malformed Z gate"
                )

                continue

            target = int(match[0])

            if target >= num_qubits:

                errors.append(
                    f"Line {line_no}: Invalid qubit index"
                )

                continue

            qc.add_gate(
                z,
                q[target]
            )

            continue


        # -------------------
        # Single-Qubit Gates
        # -------------------

        parts = line.split()

        if len(parts) != 2:

            errors.append(
                f"Line {line_no}: Malformed instruction"
            )

            continue

        gate_name = parts[0]

        if gate_name not in gate_map:

            errors.append(
                f"Line {line_no}: Unknown gate '{gate_name}'"
            )

            continue

        match = re.search(
            r'q\[(\d+)\]',
            line
        )

        if not match:

            errors.append(
                f"Line {line_no}: Missing qubit index"
            )

            continue

        qubit_index = int(
            match.group(1)
        )

        if qubit_index >= num_qubits:

            errors.append(
                f"Line {line_no}: Invalid qubit index {qubit_index}"
            )

            continue

        qc.add_gate(
            gate_map[gate_name],
            q[qubit_index]
        )

    return qc, errors


# ----------------------------------
# Example Usage
# ----------------------------------

print("Enter QASM code.")
print("Type END on a new line when finished.")

lines = []

while True:
    line = input()

    if line == "END":
        break

    lines.append(line)

qasm_code = "\n".join(lines)
r=input("Enter the QSDK version:")

circuit, errors = qasm_to_qniverse(
    qasm_code,
    r
)

if errors:

    print("Errors Found:")

    for err in errors:
        print(err)

else:

    print(
        "Conversion Successful"
    )

if not errors:
    display(circuit.draw())


Enter QASM code.
Type END on a new line when finished.


 OPENQASM 2.0;
